# Extracción de Datos desde APIs Públicas

Este notebook demuestra cómo consumir APIs REST públicas **sin autenticación** y transformar las respuestas en DataFrames listos para análisis.

| API | Datos | Endpoint base |
|-----|-------|---------------|
| **Nager.Date** | Días feriados por país y año | `https://date.nager.at/api/v3` |
| **Open-Meteo** | Clima e historial meteorológico | `https://api.open-meteo.com/v1` |

**Tecnologías:** `requests`, `pandas`, `json`

## 1. Instalación de Dependencias

In [3]:
%pip install requests pandas pyarrow

  Using cached requests-2.33.1-py3-none-any.whl.metadata (4.8 kB)
  Using cached charset_normalizer-3.4.7-cp313-cp313-win_amd64.whl.metadata (41 kB)
  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
  Using cached urllib3-2.6.3-py3-none-any.whl.metadata (6.9 kB)
  Using cached certifi-2026.2.25-py3-none-any.whl.metadata (2.5 kB)
Using cached requests-2.33.1-py3-none-any.whl (64 kB)
Using cached charset_normalizer-3.4.7-cp313-cp313-win_amd64.whl (158 kB)
Using cached idna-3.11-py3-none-any.whl (71 kB)
Using cached urllib3-2.6.3-py3-none-any.whl (131 kB)
Using cached certifi-2026.2.25-py3-none-any.whl (153 kB)

   ---------------------------------------- 0/5 [urllib3]
   ---------------------------------------- 0/5 [urllib3]
   ---------------------------------------- 0/5 [urllib3]
   -------- ------------------------------- 1/5 [idna]
   ---------------- ----------------------- 2/5 [charset_normalizer]
   -------------------------------- ------- 4/5 [requests]
   -------------


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Importación de Librerías

In [5]:
import requests
import pandas as pd
import json
from pathlib import Path

# Directorios de salida
DIR_OUTPUT = Path('datos/output')
DIR_OUTPUT.mkdir(parents=True, exist_ok=True)

print('Librerías importadas correctamente.')
print(f'Directorio de salida: {DIR_OUTPUT.resolve()}')

Librerías importadas correctamente.
Directorio de salida: C:\Users\Sergio Orozco\Downloads\clases\Ingenieria_datos\notebook\unidad_II\datos\output


---
## PARTE A — Días Feriados (Nager.Date API)

Documentación: https://date.nager.at/swagger/index.html  
No requiere API key. Soporta más de 100 países con código ISO 3166-1 alpha-2.

### A1. Países Disponibles

In [6]:

# Petición GET — países disponibles
url_paises = 'https://date.nager.at/api/v3/AvailableCountries'
respuesta = requests.get(url_paises)

# Ver la respuesta cruda
print('Status code :', respuesta.status_code)
print('Content-Type:', respuesta.headers['Content-Type'])
print('\nTexto raw (primeros 500 caracteres):')
print(respuesta.text[:500])


Status code : 200
Content-Type: application/json; charset=utf-8

Texto raw (primeros 500 caracteres):
[{"countryCode":"AD","name":"Andorra"},{"countryCode":"AL","name":"Albania"},{"countryCode":"AM","name":"Armenia"},{"countryCode":"AR","name":"Argentina"},{"countryCode":"AT","name":"Austria"},{"countryCode":"AU","name":"Australia"},{"countryCode":"AX","name":"Åland Islands"},{"countryCode":"BA","name":"Bosnia and Herzegovina"},{"countryCode":"BB","name":"Barbados"},{"countryCode":"BD","name":"Bangladesh"},{"countryCode":"BE","name":"Belgium"},{"countryCode":"BG","name":"Bulgaria"},{"countryCode


In [7]:

# Convertir el JSON a DataFrame
df_paises = pd.DataFrame(respuesta.json())
print(f'Total de países: {len(df_paises)}')
df_paises.head(10)


Total de países: 122


,countryCode,name
0,AD,Andorra
1,AL,Albania
2,AM,Armenia
3,AR,Argentina
4,AT,Austria
5,AU,Australia
6,AX,Åland Islands
7,BA,Bosnia and Herzegovina
8,BB,Barbados
9,BD,Bangladesh


### A2. Feriados de Argentina para el Año Actual

In [8]:

# Petición GET — feriados
PAIS = 'AR'
ANIO = 2026

url_feriados = f'https://date.nager.at/api/v3/PublicHolidays/{ANIO}/{PAIS}'
respuesta_f = requests.get(url_feriados)

# Ver la respuesta cruda
print('Status code :', respuesta_f.status_code)
print('URL llamada :', respuesta_f.url)
print('\nJSON raw (primeros 600 caracteres):')
print(respuesta_f.text[:600])


Status code : 200
URL llamada : https://date.nager.at/api/v3/PublicHolidays/2026/AR

JSON raw (primeros 600 caracteres):
[{"date":"2026-01-01","localName":"Año Nuevo","name":"New Year's Day","countryCode":"AR","fixed":false,"global":true,"counties":null,"launchYear":null,"types":["Public"]},{"date":"2026-02-16","localName":"Carnaval","name":"Carnival","countryCode":"AR","fixed":false,"global":true,"counties":null,"launchYear":null,"types":["Public"]},{"date":"2026-02-17","localName":"Carnaval","name":"Carnival","countryCode":"AR","fixed":false,"global":true,"counties":null,"launchYear":null,"types":["Public"]},{"date":"2026-03-24","localName":"Día Nacional de la Memoria por la Verdad y la Justicia","name":"Day o


In [9]:
# Convertir el JSON a DataFrame
df_feriados = pd.DataFrame(respuesta_f.json())
df_feriados['date'] = pd.to_datetime(df_feriados['date'])
print(f'Total de feriados en {PAIS} para {ANIO}: {len(df_feriados)}')
df_feriados

Total de feriados en AR para 2026: 16


,date,localName,name,countryCode,fixed,global,counties,launchYear,types
0,2026-01-01,Año Nuevo,New Year's Day,AR,False,True,None,None,[Public]
1,2026-02-16,Carnaval,Carnival,AR,False,True,None,None,[Public]
2,2026-02-17,Carnaval,Carnival,AR,False,True,None,None,[Public]
3,2026-03-24,Día Nacional de la Memoria por la Verdad y la ...,Day of Remembrance for Truth and Justice,AR,False,True,None,None,[Public]
4,2026-04-02,Día del Veterano y de los Caídos en la Guerra ...,Day of the Veterans and Fallen of the Malvinas...,AR,False,True,None,None,[Public]
5,2026-04-03,Viernes Santo,Good Friday,AR,False,True,None,None,[Public]
6,2026-05-01,Día del Trabajador,Labour Day,AR,False,True,None,None,[Public]
7,2026-05-25,Día de la Revolución de Mayo,May Revolution,AR,False,True,None,None,[Public]
8,2026-06-15,Paso a la Inmortalidad del General Martín Migu...,Anniversary of the Passing of General Martín M...,AR,False,True,None,None,[Public]
9,2026-06-20,Paso a la Inmortalidad del General Manuel Belg...,General Manuel Belgrano Memorial Day,AR,False,True,None,None,[Public]


### A4. Guardar Feriados en CSV y Parquet

In [10]:

# Guardar feriados en CSV y Parquet
csv_feriados     = DIR_OUTPUT / f'feriados_{PAIS}_{ANIO}.csv'
parquet_feriados = DIR_OUTPUT / f'feriados_{PAIS}_{ANIO}.parquet'

df_feriados.to_csv(csv_feriados, index=False, encoding='utf-8-sig')
df_feriados.to_parquet(parquet_feriados, index=False, engine='pyarrow')

print(f'✅ CSV     : {csv_feriados}')
print(f'✅ Parquet : {parquet_feriados}')


✅ CSV     : datos\output\feriados_AR_2026.csv
✅ Parquet : datos\output\feriados_AR_2026.parquet


---
## PARTE B — Clima e Historial Meteorológico (Open-Meteo API)

Documentación: https://open-meteo.com/en/docs  
No requiere API key. Devuelve datos históricos y pronóstico.

### B1. Pronóstico del Clima — Buenos Aires

In [11]:

# Petición GET — pronóstico del clima (Buenos Aires, próximos 7 días)
url_clima = (
    'https://api.open-meteo.com/v1/forecast'
    '?latitude=-34.6037&longitude=-58.3816'
    '&daily=temperature_2m_max,temperature_2m_min,precipitation_sum,windspeed_10m_max'
    '&timezone=America%2FArgentina%2FBuenos_Aires&forecast_days=7'
)

respuesta_c = requests.get(url_clima)

# Ver la respuesta cruda
print('Status code :', respuesta_c.status_code)
print('URL llamada :', respuesta_c.url)
print('\nJSON raw:')
print(respuesta_c.text)


Status code : 200
URL llamada : https://api.open-meteo.com/v1/forecast?latitude=-34.6037&longitude=-58.3816&daily=temperature_2m_max,temperature_2m_min,precipitation_sum,windspeed_10m_max&timezone=America%2FArgentina%2FBuenos_Aires&forecast_days=7

JSON raw:
{"latitude":-34.625,"longitude":-58.5,"generationtime_ms":0.11456012725830078,"utc_offset_seconds":-10800,"timezone":"America/Argentina/Buenos_Aires","timezone_abbreviation":"GMT-3","elevation":18.0,"daily_units":{"time":"iso8601","temperature_2m_max":"°C","temperature_2m_min":"°C","precipitation_sum":"mm","windspeed_10m_max":"km/h"},"daily":{"time":["2026-04-20","2026-04-21","2026-04-22","2026-04-23","2026-04-24","2026-04-25","2026-04-26"],"temperature_2m_max":[22.7,22.3,20.1,22.3,22.2,20.4,14.7],"temperature_2m_min":[19.5,13.8,12.5,11.1,12.8,13.6,8.3],"precipitation_sum":[5.80,8.60,0.00,0.00,0.00,0.60,0.00],"windspeed_10m_max":[15.5,17.3,14.5,9.2,6.6,18.0,19.9]}}


### B2. Clima de Varias Ciudades

In [12]:

# Convertir a DataFrame
df_clima = pd.DataFrame(respuesta_c.json()['daily'])
df_clima['time'] = pd.to_datetime(df_clima['time'])
df_clima.rename(columns={
    'time':               'fecha',
    'temperature_2m_max': 'temp_max_C',
    'temperature_2m_min': 'temp_min_C',
    'precipitation_sum':  'precipitacion_mm',
    'windspeed_10m_max':  'viento_max_kmh',
}, inplace=True)

print('Pronóstico Buenos Aires — 7 días:')
df_clima


Pronóstico Buenos Aires — 7 días:


,fecha,temp_max_C,temp_min_C,precipitacion_mm,viento_max_kmh
0,2026-04-20,22.7,19.5,5.8,15.5
1,2026-04-21,22.3,13.8,8.6,17.3
2,2026-04-22,20.1,12.5,0.0,14.5
3,2026-04-23,22.3,11.1,0.0,9.2
4,2026-04-24,22.2,12.8,0.0,6.6
5,2026-04-25,20.4,13.6,0.6,18.0
6,2026-04-26,14.7,8.3,0.0,19.9


### B3. Historial Climatológico Buenos Aires (últimos 30 días)

In [13]:

from datetime import date, timedelta

hoy      = date.today()
hace_30d = hoy - timedelta(days=30)

# Petición GET — historial climatológico (últimos 30 días)
url_hist = (
    f'https://archive-api.open-meteo.com/v1/archive'
    f'?latitude=-34.6037&longitude=-58.3816'
    f'&start_date={hace_30d}&end_date={hoy}'
    f'&daily=temperature_2m_max,temperature_2m_min,precipitation_sum'
    f'&timezone=America%2FArgentina%2FBuenos_Aires'
)

respuesta_h = requests.get(url_hist)

# Ver la respuesta cruda
print('Status code :', respuesta_h.status_code)
print('URL llamada :', respuesta_h.url)
print('\nJSON raw:')
print(respuesta_h.text)


Status code : 200
URL llamada : https://archive-api.open-meteo.com/v1/archive?latitude=-34.6037&longitude=-58.3816&start_date=2026-03-21&end_date=2026-04-20&daily=temperature_2m_max,temperature_2m_min,precipitation_sum&timezone=America%2FArgentina%2FBuenos_Aires

JSON raw:
{"latitude":-34.622143,"longitude":-58.40909,"generationtime_ms":1.1780261993408203,"utc_offset_seconds":-10800,"timezone":"America/Argentina/Buenos_Aires","timezone_abbreviation":"GMT-3","elevation":18.0,"daily_units":{"time":"iso8601","temperature_2m_max":"°C","temperature_2m_min":"°C","precipitation_sum":"mm"},"daily":{"time":["2026-03-21","2026-03-22","2026-03-23","2026-03-24","2026-03-25","2026-03-26","2026-03-27","2026-03-28","2026-03-29","2026-03-30","2026-03-31","2026-04-01","2026-04-02","2026-04-03","2026-04-04","2026-04-05","2026-04-06","2026-04-07","2026-04-08","2026-04-09","2026-04-10","2026-04-11","2026-04-12","2026-04-13","2026-04-14","2026-04-15","2026-04-16","2026-04-17","2026-04-18","2026-04-19","202

### B4. Guardar Clima en CSV y Parquet

In [14]:

# Convertir a DataFrame
df_hist = pd.DataFrame(respuesta_h.json()['daily'])
df_hist['time'] = pd.to_datetime(df_hist['time'])
df_hist.rename(columns={
    'time':               'fecha',
    'temperature_2m_max': 'temp_max_C',
    'temperature_2m_min': 'temp_min_C',
    'precipitation_sum':  'precipitacion_mm',
}, inplace=True)

print(f'Historial Buenos Aires: {hace_30d} → {hoy} ({len(df_hist)} días)')
df_hist


Historial Buenos Aires: 2026-03-21 → 2026-04-20 (31 días)


,fecha,temp_max_C,temp_min_C,precipitacion_mm
0,2026-03-21,23.7,16.3,47.4
1,2026-03-22,23.4,13.1,0.0
2,2026-03-23,24.1,16.5,0.2
3,2026-03-24,24.5,11.6,0.0
4,2026-03-25,25.9,14.0,0.0
5,2026-03-26,25.8,17.2,0.5
6,2026-03-27,24.8,20.4,6.2
7,2026-03-28,25.1,21.5,3.9
8,2026-03-29,26.2,21.5,26.8
9,2026-03-30,27.1,22.0,0.1



---
## PARTE C — Guardar y Resumen de Archivos


In [15]:

# Guardar clima y feriados en CSV y Parquet
df_clima.to_csv(DIR_OUTPUT / 'clima_buenosaires_7dias.csv', index=False, encoding='utf-8-sig')
df_clima.to_parquet(DIR_OUTPUT / 'clima_buenosaires_7dias.parquet', index=False, engine='pyarrow')

df_hist.to_csv(DIR_OUTPUT / f'clima_buenosaires_hist_{hace_30d}_{hoy}.csv', index=False, encoding='utf-8-sig')
df_hist.to_parquet(DIR_OUTPUT / f'clima_buenosaires_hist_{hace_30d}_{hoy}.parquet', index=False, engine='pyarrow')

df_feriados.to_csv(DIR_OUTPUT / f'feriados_{PAIS}_{ANIO}.csv', index=False, encoding='utf-8-sig')
df_feriados.to_parquet(DIR_OUTPUT / f'feriados_{PAIS}_{ANIO}.parquet', index=False, engine='pyarrow')

# Resumen de archivos generados
archivos = [
    {'archivo': f.name, 'formato': f.suffix, 'tamaño_KB': round(f.stat().st_size / 1024, 2)}
    for f in sorted(DIR_OUTPUT.iterdir())
]
print('Archivos en datos/output:')
pd.DataFrame(archivos)


Archivos en datos/output:


,archivo,formato,tamaño_KB
0,clima_buenosaires_7dias.csv,.csv,0.27
1,clima_buenosaires_7dias.parquet,.parquet,3.73
2,clima_buenosaires_hist_2026-03-21_2026-04-20.csv,.csv,0.84
3,clima_buenosaires_hist_2026-03-21_2026-04-20.p...,.parquet,3.58
4,feriados_AR_2026.csv,.csv,1.61
5,feriados_AR_2026.parquet,.parquet,6.09
6,person_person.csv,.csv,917.14
7,person_person.parquet,.parquet,241.73
